# EDA — Part C: Image Grids + Pixel-Intensity / Brightness / Contrast

**Goal:** Actually *look* at the data. Render sample face grids per emotion and
analyse pixel-intensity statistics — the global histogram, and per-image
**brightness** (mean) and **contrast** (std) distributions.

**Why this matters:** in-the-wild faces vary wildly in lighting. Some images are
too dark, too bright, or low-contrast. Seeing that spread is what *justifies* the
preprocessing choices in `config.yaml → preprocessing.normalization`
(`none | rescale | standardize | histogram_eq`), implemented in issues #20–#21.

**Key definitions:**
- **Brightness** ≈ mean intensity of an image (how light/dark overall).
- **Contrast** ≈ standard deviation of intensities (how much light-to-dark spread).

Sampling is **seeded** (`config.yaml → global.seed`) so the grids are reproducible.

## 0. Setup

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

ROOT = Path().resolve().parent  # notebooks/ → project root
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.emotion_detector.utils.config import load_config
from src.emotion_detector.utils.logging import setup_logging
from src.emotion_detector.data.fer2013 import Fer2013Fetcher

cfg = load_config(ROOT / "config.yaml")
for key in cfg["paths"]:
    cfg["paths"][key] = str(ROOT / cfg["paths"][key])
setup_logging(cfg)

SEED = cfg["global"]["seed"]
rng = np.random.default_rng(SEED)  # seeded → reproducible sample grids

EDA_DIR = Path(cfg["paths"]["results_dir"]) / "eda"
EDA_DIR.mkdir(parents=True, exist_ok=True)

EMOTION_LABELS = {
    0: "Angry", 1: "Disgust", 2: "Fear",
    3: "Happy", 4: "Sad",     5: "Surprise", 6: "Neutral",
}

print(f"Seed = {SEED}; figures → {EDA_DIR}")

In [ ]:
def show_grid(images, titles, rows, cols, suptitle, save_name=None):
    """Render a grid of grayscale images with per-cell titles.

    vmin/vmax are fixed to 0/255 so brightness is comparable across cells
    (imshow would otherwise auto-scale each image independently).
    """
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 1.6, rows * 1.9))
    axes = np.atleast_1d(axes).ravel()
    for ax, img, title in zip(axes, images, titles):
        ax.imshow(img, cmap="gray", vmin=0, vmax=255)
        ax.set_title(title, fontsize=8)
        ax.axis("off")
    for ax in axes[len(images):]:      # blank any unused cells
        ax.axis("off")
    fig.suptitle(suptitle, fontsize=13)
    plt.tight_layout()
    if save_name:
        path = EDA_DIR / save_name
        fig.savefig(path, dpi=120, bbox_inches="tight")
        print(f"Saved: {path}")
    plt.show()

## 1. Load the training images

In [ ]:
fetcher = Fer2013Fetcher(cfg)
images, labels = fetcher.fetch("Training")
print(f"images: {images.shape} {images.dtype}   labels: {labels.shape}")

## 2. Random sample grid (5 × 7)

A first look at 35 random faces, each labelled with its emotion. This turns the
`(48, 48)` arrays back into viewable images via `imshow(cmap="gray")`.

In [ ]:
idx = rng.choice(len(images), size=35, replace=False)
show_grid(
    [images[i] for i in idx],
    [EMOTION_LABELS[labels[i]] for i in idx],
    rows=5, cols=7,
    suptitle="Random training samples",
    save_name="sample_grid_random.png",
)

## 3. Per-emotion sample grid

One row per emotion (8 random samples each). This makes class-specific quality
issues visible — e.g. whether *Disgust* (the rare class) looks consistent, or
whether some classes have more dark / occluded / non-face crops.

In [ ]:
per_class = 8
rows, cols = len(EMOTION_LABELS), per_class
fig, axes = plt.subplots(rows, cols, figsize=(cols * 1.5, rows * 1.7))

for r, code in enumerate(sorted(EMOTION_LABELS)):
    class_idx = np.where(labels == code)[0]
    picks = rng.choice(class_idx, size=per_class, replace=False)
    for c, i in enumerate(picks):
        ax = axes[r, c]
        ax.imshow(images[i], cmap="gray", vmin=0, vmax=255)
        ax.axis("off")
    axes[r, 0].set_ylabel(EMOTION_LABELS[code], rotation=0, ha="right",
                          va="center", fontsize=10)
    # re-enable the y-axis label we just set (axis('off') hid it)
    axes[r, 0].axis("on")
    axes[r, 0].set_xticks([]); axes[r, 0].set_yticks([])
    for spine in axes[r, 0].spines.values():
        spine.set_visible(False)

fig.suptitle("Random samples per emotion (one row each)", fontsize=13)
plt.tight_layout()
fig.savefig(EDA_DIR / "sample_grid_per_emotion.png", dpi=120, bbox_inches="tight")
print(f"Saved: {EDA_DIR / 'sample_grid_per_emotion.png'}")
plt.show()

## 4. Global pixel-intensity histogram

Flatten every image into one long vector and histogram all ~66M pixel values.
This shows the overall lighting profile of the dataset. `skew` and `kurtosis`
quantify the asymmetry and tailedness we eyeballed in `01_eda.ipynb` (mean < median).

In [ ]:
flat = images.reshape(-1).astype(np.float32)  # (N*48*48,)

mean_v   = flat.mean()
median_v = np.median(flat)
skew_v   = float(stats.skew(flat))
kurt_v   = float(stats.kurtosis(flat))  # Fisher: 0 == normal

print(f"mean     : {mean_v:.2f}")
print(f"median   : {median_v:.1f}")
print(f"std      : {flat.std():.2f}")
print(f"skewness : {skew_v:+.3f}  (negative → tail toward dark)")
print(f"kurtosis : {kurt_v:+.3f}  (Fisher; 0 = normal, <0 = flatter)")

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(flat, bins=64, range=(0, 255), color="steelblue", edgecolor="white")
ax.axvline(mean_v, color="red", ls="--", label=f"mean {mean_v:.0f}")
ax.axvline(median_v, color="orange", ls="--", label=f"median {median_v:.0f}")
ax.set_title("Global pixel-intensity distribution (all training images)")
ax.set_xlabel("Pixel intensity (0–255)")
ax.set_ylabel("Pixel count")
ax.legend()
plt.tight_layout()
fig.savefig(EDA_DIR / "intensity_histogram_global.png", dpi=120, bbox_inches="tight")
print(f"Saved: {EDA_DIR / 'intensity_histogram_global.png'}")
plt.show()

## 5. Per-image brightness and contrast distributions

Collapse each image to two numbers:
- **brightness** = `image.mean()` over the 48×48 grid → one value per image.
- **contrast**  = `image.std()`  over the 48×48 grid → one value per image.

The *spread* of these per-image values is the lighting variance across the dataset.

In [ ]:
brightness = images.mean(axis=(1, 2))  # (N,) — one mean per image
contrast   = images.std(axis=(1, 2))   # (N,) — one std per image

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(brightness, bins=50, color="darkorange", edgecolor="white")
axes[0].set_title(f"Per-image brightness (mean)\nμ={brightness.mean():.1f}, σ={brightness.std():.1f}")
axes[0].set_xlabel("Mean intensity"); axes[0].set_ylabel("Number of images")

axes[1].hist(contrast, bins=50, color="seagreen", edgecolor="white")
axes[1].set_title(f"Per-image contrast (std)\nμ={contrast.mean():.1f}, σ={contrast.std():.1f}")
axes[1].set_xlabel("Std of intensity"); axes[1].set_ylabel("Number of images")

plt.tight_layout()
fig.savefig(EDA_DIR / "brightness_contrast_hist.png", dpi=120, bbox_inches="tight")
print(f"Saved: {EDA_DIR / 'brightness_contrast_hist.png'}")
plt.show()

In [ ]:
# Brightness vs contrast — each dot is one image. Dark or low-contrast images
# cluster toward the axes; a healthy face sits in the middle cloud.
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(brightness, contrast, s=3, alpha=0.15, color="navy")
ax.set_title("Brightness vs contrast (one dot per training image)")
ax.set_xlabel("Brightness (mean intensity)")
ax.set_ylabel("Contrast (std intensity)")
plt.tight_layout()
fig.savefig(EDA_DIR / "brightness_vs_contrast_scatter.png", dpi=120, bbox_inches="tight")
print(f"Saved: {EDA_DIR / 'brightness_vs_contrast_scatter.png'}")
plt.show()

## 6. Flag suspicious images (dark / bright / low-contrast)

Using the candidate thresholds from `01_eda.ipynb` §13. These counts and the
example grids below are the empirical case for normalization / histogram
equalization in preprocessing.

In [ ]:
DARK_T, BRIGHT_T, LOWC_T = 40, 215, 15

dark_idx      = np.where(brightness < DARK_T)[0]
bright_idx    = np.where(brightness > BRIGHT_T)[0]
lowc_idx      = np.where(contrast < LOWC_T)[0]
constant_idx  = np.where(contrast == 0)[0]

n = len(images)
print(f"Total training images        : {n:,}")
print(f"Dark      (brightness < {DARK_T}) : {len(dark_idx):>5,}  ({len(dark_idx)/n:.2%})")
print(f"Bright    (brightness > {BRIGHT_T}): {len(bright_idx):>5,}  ({len(bright_idx)/n:.2%})")
print(f"Low-contrast (std < {LOWC_T})      : {len(lowc_idx):>5,}  ({len(lowc_idx)/n:.2%})")
print(f"Constant  (std == 0)          : {len(constant_idx):>5,}  ({len(constant_idx)/n:.2%})")

In [ ]:
# Visualise the extremes: 7 darkest, 7 brightest, 7 lowest-contrast images.
darkest  = np.argsort(brightness)[:7]
brightest = np.argsort(brightness)[-7:]
lowest_c = np.argsort(contrast)[:7]

show_grid(
    [images[i] for i in darkest],
    [f"{EMOTION_LABELS[labels[i]]}\nb={brightness[i]:.0f}" for i in darkest],
    rows=1, cols=7, suptitle="7 darkest images", save_name="flagged_darkest.png",
)
show_grid(
    [images[i] for i in brightest],
    [f"{EMOTION_LABELS[labels[i]]}\nb={brightness[i]:.0f}" for i in brightest],
    rows=1, cols=7, suptitle="7 brightest images", save_name="flagged_brightest.png",
)
show_grid(
    [images[i] for i in lowest_c],
    [f"{EMOTION_LABELS[labels[i]]}\nc={contrast[i]:.0f}" for i in lowest_c],
    rows=1, cols=7, suptitle="7 lowest-contrast images", save_name="flagged_lowcontrast.png",
)

## 7. Findings for `data.md` §2 / §3

Fill the **Result** column from the outputs above after running on the real data.

| Observation | Result | Motivates |
|---|---|---|
| Global intensity mean / median | — / — | — |
| Global skewness (asymmetry) | — | left-skew = tail toward dark |
| Global kurtosis (tailedness) | — | — |
| Per-image brightness spread (σ) | — | wide spread → lighting varies a lot |
| Per-image contrast spread (σ) | — | wide spread → some flat/washed-out |
| Dark images (brightness < 40) | — | `preprocessing.normalization` |
| Bright images (brightness > 215) | — | `preprocessing.normalization` |
| Low-contrast images (std < 15) | — | `histogram_eq` / `standardize` |
| Constant images (std == 0) | — | cleaning: drop degenerate images |

**Implications for a CNN (learning-loop discussion):**
- A CNN's first-layer filters respond to *local intensity gradients*. If lighting
  varies wildly image-to-image, the same expression produces very different raw
  pixel values, so the network wastes capacity learning to be lighting-invariant.
- **Rescale** (÷255) fixes the *scale* so gradients are well-conditioned, but not
  the *per-image* lighting differences.
- **Standardize** (per-image or per-dataset mean/std) removes brightness offset.
- **Histogram equalization** redistributes intensities to use the full 0–255 range,
  directly attacking the low-contrast images flagged above.

These are exactly the toggleable options in `config.yaml → preprocessing.normalization`
(`none | rescale | standardize | histogram_eq`), ablated in issues #20–#21.